In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import torch

from src.config import SEED
from src.transfer import build_feature_extractor, train_transfer

torch.manual_seed(SEED)

# Transfer Learning con ResNet50
## Fase 4 - Feature Extraction

En lugar de entrenar una red desde cero como la CNN, aprovechamos **ResNet50 preentrenada en ImageNet**: una red que ya aprendió a reconocer bordes, texturas y estructuras visuales complejas en 1.2 millones de imágenes.

Usamos ImageNet específicamente porque normalizamos nuestras imágenes con sus stats (mean y std), lo que garantiza que la red recibe los datos en el rango que espera.

**Estrategia: Feature Extraction**  
El backbone de ResNet50 queda completamente congelado. Solo se entrena el clasificador final (fc), que reemplaza la cabeza original (2048 → 1000 clases ImageNet) por una binaria (2048 $\rightarrow$ 1 logit).

| Qué se entrena | LR | Parámetros entrenables |
|---|---|---|
| Solo clasificador (fc) | 1e-3 | aprox 2049 |

In [ ]:
model = build_feature_extractor()

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parámetros totales:      {total:,}")
print(f"Parámetros entrenables:  {trainable:,}  ({100*trainable/total:.1f}%)")

In [ ]:
historial = train_transfer(model)